# ISAN 3392 - Model A: predicting the smallest safe GPU slice
**Alibaba ASI trace (OSDI '26) - D1_RIGHTSIZING.** Covers plan phases 3-6: baseline go/no-go, three model families, LOCO, conformal safety margin, interpretation.

**How to use (Colab)**
1. Put your D1 parquet export (and, optionally, the D2 export, used only to read the list of deployable slice sizes) in one Drive folder.
2. Set `DATA_DIR` in the *Setup and configuration* section, then **Runtime > Run all**. Free-tier CPU is enough; no GPU needed.
3. First thing to read: the **schema report** printed when D1 is loaded. It tells you which columns of *your* export differ from the plan.
4. The decision point is the **GO / NO-GO checkpoint** section: if no learned model beats the percentile baseline A1 on held-out clusters, stop and write the negative result.

**What this notebook was tested on:** an end-to-end run on a *synthetic* stand-in with the plan's schema (so the code paths, shapes and metrics are verified).
It has **not** been run on your real data - numbers on the real data are yours to find. Set `ISAN_FAST=1` only for smoke tests.

**Design choices that differ from the plan (and why)**
* A3 uses **XGBoost** (`reg:quantileerror` + monotone constraints). LightGBM refuses `monotone_constraints` with the quantile objective.
* Conformal level is **alpha = 0.05**, so coverage matches the plan's headline `U <= 5%`.
* A1's history uses only pods whose label was **complete at admission time**; the plan's literal "prior admission days" rule is run alongside as a leakage audit.
* `cluster_id` / `asw_id` are not features in LOCO runs (an unseen cluster has unseen ids).
* A2 is fitted with a smoothed pinball loss (L-BFGS) because sklearn's exact `QuantileRegressor` does not scale to ~200k rows.

## 1. Setup and configuration
Run this cell first. It mounts Google Drive (in Colab), installs anything missing, and holds **every setting you may want to change**.
Nothing below this block should need editing unless a check cell tells you a column name differs from the plan.

In [ ]:
import os, sys, glob, fnmatch, subprocess, warnings, time, json, math
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:                                       # Colab already ships most of these; this only adds/upgrades what is missing
    import importlib.util                          # install only what is missing, so pip never swaps a library under a running session
    _need = [p for p, m in (("xgboost>=2.0", "xgboost"), ("lightgbm", "lightgbm"), ("shap", "shap")) if importlib.util.find_spec(m) is None]
    if _need: subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=False)
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import matplotlib
try: get_ipython(); _IN_NOTEBOOK = True          # Colab / Jupyter: keep the inline backend so figures show
except NameError: _IN_NOTEBOOK = False; matplotlib.use("Agg")   # plain-python run: draw to files only
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy import stats

# ------------------------------ CONFIG ------------------------------
FAST = os.environ.get("ISAN_FAST", "0") == "1"      # 1 = smoke-test profile with tiny budgets. Leave 0 for real runs.
DATA_DIR = os.environ.get("ISAN_DATA_DIR", "/content/drive/MyDrive/ISAN3392")   # folder holding the D1 / D2 parquet exports
OUT_DIR = os.environ.get("ISAN_OUT_DIR", os.path.join(DATA_DIR, "results"))     # models, tables and figures are written here
D1_PATTERN, D2_PATTERN = "*D1*", "*D2*"             # name patterns (file or folder, case-insensitive) inside DATA_DIR
SEED = 3392
TRAIN_DAYS, VAL_DAYS, TEST_DAYS = (14, 129), (130, 157), (158, 183)      # plan section 4.1 (admission day)
MIN_CLUSTER_PODS = 1000 if FAST else 9000           # plan 4.3: clusters with >= 9,000 eligible D1 pods become LOCO folds
ALPHA = 0.05                                        # conformal miscoverage. 0.05 matches the headline U <= 5% (the plan's 90% does not)
TAU_MAIN = 0.90                                     # operating quantile of the base model
HIST_WINDOWS = (3, 7, 14)                           # A1 sliding-window lengths in days
N_JOBS = -1                                         # all cores
FIG_DPI = 110
os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)
print(f"FAST={FAST} | IN_COLAB={IN_COLAB} | DATA_DIR={DATA_DIR}")
print("files found:", sorted(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else "DATA_DIR NOT FOUND - fix DATA_DIR above")
print("xgboost", xgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
assert tuple(int(x) for x in xgb.__version__.split(".")[:2]) >= (2, 0), "need xgboost >= 2.0 for reg:quantileerror"

## 2. Schema, leakage gate and loader
The feature lists below are copied from plan sections 3.3 and 3.4. **If your real export uses different column names,
the schema report printed when the data is loaded will show exactly which ones, and you only change the lists here.**

In [ ]:
NUMERIC = ["gpu_request", "vram_entitled_gib", "cpu_request_cores", "server_gpu_count", "server_cpu_capacity_cores",
           "gpu_mem_per_gpu", "cpu_per_gpu", "adm_hour", "adm_dow_idx", "hour_sin", "hour_cos", "dow_sin", "dow_cos",
           "is_whole_gpu", "is_fractional", "is_multi_gpu", "has_workload_id",
           "cluster_pods_at_submit", "cluster_gpu_requested_at_submit", "cluster_sm_mean_prev_hr", "cluster_sm_p90_prev_hr",
           "server_alloc_gpu_at_submit", "server_gpu_occupancy_at_submit", "asw_pods_at_submit", "asw_alloc_gpu_at_submit",
           "wl_hist_pod_count", "wl_hist_days_observed", "wl_hist_mvf_p90", "wl_hist_mvf_max", "wl_hist_mvf_mean",
           "wl_hist_sm_p90", "wl_hist_dur_mean", "is_genai_request"]
CATEG = ["gpu_spec_public", "priority_class", "model_type_public"]     # categoricals that transfer across clusters
CATEG_CLUSTER_SPECIFIC = ["cluster_id", "asw_id"]                       # cannot generalise to an unseen cluster -> excluded from LOCO models
HIST_COLS = [c for c in NUMERIC if c.startswith("wl_hist_")] + ["has_workload_id"]
CLUSTER_LOAD_COLS = ["cluster_pods_at_submit", "cluster_gpu_requested_at_submit", "cluster_sm_mean_prev_hr", "cluster_sm_p90_prev_hr"]
# Plan 3.4: derived from observed utilisation -> a build gate fails if any of these is used as a predictor
LEAKY = {"avg_gpu_sm_util", "avg_gpu_mem_gib", "used_gpu_hours", "sm_p90_norm", "sm_mean_norm", "mem_frac_p90",
         "peak_to_mean", "waste_tier", "wasted_gpu_hours", "min_viable_fraction"}
UNKNOWN_AT_ADMISSION = {"n_pod_hours", "n_hours_with_util", "is_censored", "schedule_delay_sec", "ready_delay_sec", "duration_hours"}
TARGETS = {"y_mvf", "y_ready", "y_ready_jes_crosscheck"}
# Monotone constraints (A3). Only the history features: "more historical demand never lowers the slice" is a property
# you can state in one sentence. cpu_per_gpu is NOT constrained: the OSDI'26 paper (3.4) shows CTR/rec pods are CPU-heavy but low-SM,
# so its sign is probably negative across segments. Section 7 prints the within-segment sign so you can decide with evidence.
MONO = {"wl_hist_mvf_p90": 1, "wl_hist_mvf_max": 1, "wl_hist_mvf_mean": 1, "wl_hist_sm_p90": 1}
DAY_CANDIDATES = ["adm_day", "admission_day", "first_day", "day"]
DUR_CANDIDATES = ["duration_hours", "n_pod_hours", "n_hours_with_util", "duration_h", "duration_hrs", "observed_hours", "duration"]   # how long each pod was observed (hours)
WL_CANDIDATES = ["workload_code", "workload_id_code", "wl_code", "workload_id"]

def _resolve(path_cols, wanted):
    low = {c.lower(): c for c in path_cols}
    return [low[w.lower()] for w in wanted if w.lower() in low]

def read_parquet_any(path, columns=None):
    """Read a parquet file or a folder of parts. DECIMAL -> float64 at the Arrow level, before pandas ever sees it
    (Snowflake NUMBER otherwise arrives as Python Decimal objects: 13.6x more RAM, see plan 6.5). Column names are lower-cased."""
    files = sorted(glob.glob(os.path.join(path, "**", "*.parquet"), recursive=True)) if os.path.isdir(path) else [path]
    if not files: raise FileNotFoundError(f"no .parquet under {path}")
    parts = []
    for f in files:
        cols = _resolve(pq.read_schema(f).names, columns) if columns else None
        t = pq.read_table(f, columns=cols)
        for i, fld in enumerate(t.schema):
            if pa.types.is_decimal(fld.type): t = t.set_column(i, fld.name, t.column(i).cast(pa.float64()))
        d = t.to_pandas(); d.columns = [c.lower() for c in d.columns]; parts.append(d)
    return pd.concat(parts, ignore_index=True)

def find_path(pattern):
    hits = [p for p in sorted(os.listdir(DATA_DIR)) if fnmatch.fnmatch(p.lower(), pattern.lower())
            and (p.lower().endswith(".parquet") or os.path.isdir(os.path.join(DATA_DIR, p)))]      # skip dictionaries, loaders, csv notes
    if not hits: raise FileNotFoundError(f"nothing matching {pattern!r} in {DATA_DIR}. Found: {sorted(os.listdir(DATA_DIR))}")
    return os.path.join(DATA_DIR, hits[0])

def pick_col(df, candidates, what):
    for c in candidates:
        if c in df.columns: return c
    raise KeyError(f"cannot find the {what} column. Tried {candidates}. Columns present: {list(df.columns)}")
print(f"{len(NUMERIC)} numeric + {len(CATEG)} transferable categorical features; {len(LEAKY)} leaky and {len(UNKNOWN_AT_ADMISSION)} unknown-at-admission columns are barred")

## 3. Data-preparation helpers: `prepare`, schema report, leakage gate
These functions are called when each dataset is loaded below. `prepare()` does what plan 6.5 asks of the loader (Decimal -> float, categoricals, int group codes) and adds the helper columns
used everywhere else. **Read the printed report**: it lists plan columns that are missing and columns the plan does not mention.

In [ ]:
GROUP_FOR_MISSING = "shape"   # pods with NO workload_id: "shape" = group near-duplicates by resource-request signature, "unique" = one group per pod
                              # (never give them all one shared code: GroupKFold would put 67% of the data in a single fold, see review item 4)

def prepare(df, target, name):
    df = df.copy()
    day_col = pick_col(df, DAY_CANDIDATES, "admission-day")
    wl_col = pick_col(df, WL_CANDIDATES, "workload id / code")
    tgt = pick_col(df, [target], "target")
    df["adm_day_"] = df[day_col].astype("int32")
    s = df[wl_col]
    if pd.api.types.is_numeric_dtype(s): s = s.where(s >= 0)           # numeric codes: negative = missing
    codes = pd.factorize(s)[0].astype("int64")                         # -1 where missing
    has = codes >= 0
    n = len(df)
    if GROUP_FOR_MISSING == "unique":
        miss_group = codes.max() + 1 + np.arange(n)
    else:
        key = df[[c for c in ["cluster_id", "gpu_spec_public", "gpu_request", "cpu_request_cores", "vram_entitled_gib", "priority_class", "model_type_public"] if c in df.columns]]
        miss_group = codes.max() + 1 + (pd.util.hash_pandas_object(key, index=False).values % 10**9).astype("int64")
    df["wl_code"] = codes; df["group"] = np.where(has, codes, miss_group)
    df["t_adm"] = df["adm_day_"].astype("float64") * 24 + df["adm_hour"].astype("float64")
    dur_col = next((c for c in DUR_CANDIDATES if c in df.columns), None)
    if dur_col:
        df["obs_hours"] = df[dur_col].astype("float64").fillna(24.0); print(f"  pod duration taken from column '{dur_col}'")
    else:
        df["obs_hours"] = 24.0
        print("  WARNING: no duration column found (tried %s).\n"
              "           Assuming every pod ran 24 h (the D1 minimum). The A1 history and the leakage audit are then approximate:\n"
              "           re-export D1/D2 with duration_hours (or n_pod_hours) for exact results." % DUR_CANDIDATES)
    df["t_end"] = df["t_adm"] + df["obs_hours"]                       # when the pod's label became complete
    if "reserved_gpu_hours" not in df.columns:
        df["reserved_gpu_hours"] = df["gpu_request"].astype("float64") * df["obs_hours"]
        print("  note: reserved_gpu_hours not in export -> computed as gpu_request * observed hours")
    for c in CATEG + CATEG_CLUSTER_SPECIFIC:
        if c in df.columns: df[c] = df[c].astype(str).astype("category")
    for c in NUMERIC:
        if c in df.columns: df[c] = df[c].astype("float32")
    df["y"] = df[tgt].astype("float64")
    df["segment"] = np.select([df["adm_day_"] < TRAIN_DAYS[0], df["adm_day_"] <= TRAIN_DAYS[1], df["adm_day_"] <= VAL_DAYS[1], df["adm_day_"] <= TEST_DAYS[1]],
                              ["warmup", "train", "val", "test"], "tail")
    df["cold"] = df["wl_hist_pod_count"].isna() if "wl_hist_pod_count" in df.columns else True
    df["is_rec"] = (df["model_type_public"].astype(str) == "rec")
    df["stratum"] = np.where(df["is_rec"], "rec", "genai") + "|" + np.where(df["cold"], "cold", "warm")
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols | workload id present {has.mean():.1%} | largest group = {np.bincount(pd.factorize(df['group'])[0]).max() / n:.2%} of rows")
    return df

def schema_report(df):
    global NUM_F, CAT_F, CLS_F
    NUM_F = [c for c in NUMERIC if c in df.columns]; CAT_F = [c for c in CATEG if c in df.columns]
    CLS_F = [c for c in CATEG_CLUSTER_SPECIFIC if c in df.columns]
    miss = [c for c in NUMERIC + CATEG if c not in df.columns]
    known = set(NUMERIC) | set(CATEG) | set(CATEG_CLUSTER_SPECIFIC) | LEAKY | UNKNOWN_AT_ADMISSION | TARGETS | set(DAY_CANDIDATES) | set(WL_CANDIDATES)
    helper = {"adm_day_", "wl_code", "group", "t_adm", "t_end", "obs_hours", "y", "segment", "cold", "is_rec", "stratum", "reserved_gpu_hours", "pod_id"}
    unk = [c for c in df.columns if c not in known and c not in helper]
    print(f"features available: {len(NUM_F)} numeric + {len(CAT_F)} categorical (+ cluster-specific: {CLS_F})")
    if miss: print("!! plan columns NOT in your export (dropped from the feature lists):", miss)
    if unk: print("?? columns the plan does not classify (NOT used - add to NUMERIC/CATEG only if they are known at admission):", unk)

def leakage_gate(features, df, target_col="y", sample=200_000):
    bad = (set(features) & (LEAKY | UNKNOWN_AT_ADMISSION | TARGETS))
    assert not bad, f"LEAKAGE GATE FAILED - barred columns used as predictors: {sorted(bad)}"
    s = df.sample(min(sample, len(df)), random_state=SEED); hard, soft = [], []
    for c in features:
        if str(s[c].dtype) == "category": continue
        v = s[[c, target_col]].dropna()
        if len(v) > 500 and v[c].nunique() > 1:
            r = abs(stats.spearmanr(v[c], v[target_col])[0])
            if r > 0.97: hard.append((c, round(float(r), 3)))
            elif r > 0.85: soft.append((c, round(float(r), 3)))
    assert not hard, f"SINGLE-FEATURE LEAKAGE SCREEN: |Spearman| > 0.97 with the target for {hard} - almost certainly a leaked column, inspect before training"
    if soft: print("  warning: very strong single features (|Spearman| 0.85-0.97):", soft, "- expected for wl_hist_* on warm rows, but this is exactly where leakage hides: see the A1 history audit in notebook A")
    print(f"leakage gate passed for {len(features)} features (barred-column check + single-feature screen)")

## 4. Splits: temporal, grouped, leave-one-cluster-out (LOCO)
* **Temporal** (plan 4.1): train = days 14-129, validation = 130-157, test = 158-183, by admission day.
* The validation segment is split into two **workload-disjoint halves** (by group parity): one for early stopping / model selection,
  one for conformal or isotonic calibration. Using the same rows for both would make the calibration optimistic.
* **LOCO protocol used throughout** (strict version): for held-out cluster *k* the model is trained on the *other* clusters' train days,
  tuned and calibrated on the *other* clusters' validation days, and scored on cluster *k*'s **test days only**. It is therefore unseen in
  both cluster and time. Hyper-parameters are chosen once on the pooled temporal validation fold and reused (equal budget, no per-fold tuning).

In [ ]:
def make_masks(df):
    seg = df["segment"].values; par = (df["group"].values % 2) == 0
    m = {s: seg == s for s in ("train", "val", "test")}
    m["val_es"] = m["val"] & par; m["val_cal"] = m["val"] & ~par
    return m

def eligible_clusters(cluster_series):
    vc = cluster_series.astype(str).value_counts()
    return [c for c in vc.index if vc[c] >= MIN_CLUSTER_PODS]

def loco_masks(df, masks, cluster):
    other = (df["cluster_id"].astype(str) != cluster).values; this = ~other
    return dict(train=masks["train"] & other, es=masks["val_es"] & other, cal=masks["val_cal"] & other, test=masks["test"] & this)

def fold_table(df, clusters):
    rows = []
    for c in clusters:
        a = df[df["cluster_id"].astype(str) == c]; t = a[a["segment"] == "test"]
        rows.append(dict(cluster=c, n_all=len(a), n_test=len(t), rec_share=round(float(a["is_rec"].mean()), 3),
                         y_p50=round(float(a["y"].median()), 3), warm_share_test=round(float(1 - t["cold"].mean()), 3) if len(t) else np.nan))
    return pd.DataFrame(rows).set_index("cluster")

## 5. Metrics and the conformal / slice-snapping machinery
* **Pinball loss** at tau, **empirical coverage**, and the plan's two operational numbers: **U** (share of pods whose recommended slice is below
  their label) and **R** (reclaimed GPU-hours). R is reported as a **share of reserved GPU-hours** and computed as `reserved_hours x (1 - slice_fraction)`,
  which fixes the double-counted request units in plan 6.1.
* One-sided **split-conformal** margin: `q = the ceil((n+1)(1-alpha))/n quantile of (y - yhat)` on the calibration half, then `slice = snap_up(yhat + q)`.
  A **Mondrian** (stratified) version calibrates separately per model_type x cold/warm, because one global margin over-covers easy strata and under-covers hard ones.
  Split-conformal assumes exchangeability, which temporal and cluster shift break - so coverage on unseen clusters is something we *measure*, not a guarantee.

In [ ]:
def pinball(y, p, tau):
    d = np.asarray(y) - np.asarray(p); return float(np.mean(np.maximum(tau * d, (tau - 1) * d)))

def conformal_margin(y_cal, p_cal, alpha):
    r = np.sort(np.asarray(y_cal) - np.asarray(p_cal)); n = len(r)
    if n == 0: return 0.0
    k = int(math.ceil((n + 1) * (1 - alpha))); return float(r[min(k, n) - 1])

def mondrian_margins(y_cal, p_cal, strata_cal, alpha, min_n=200):
    g = conformal_margin(y_cal, p_cal, alpha); out = {}
    strata_cal = np.asarray(strata_cal)
    for s in np.unique(strata_cal):
        m = strata_cal == s; out[s] = conformal_margin(y_cal[m], p_cal[m], alpha) if m.sum() >= min_n else g
    return g, out

def upper_bound(p, strata=None, g=0.0, per_stratum=None):
    q = np.full(len(p), g, dtype="float64")
    if per_stratum is not None and strata is not None: q = pd.Series(np.asarray(strata)).map(per_stratum).fillna(g).values
    return np.clip(np.asarray(p) + q, 0, 1)

def slice_fraction(p_up, req, ladder):
    """snap_up: smallest deployable size >= predicted need (always upward), never above the original request."""
    req = np.asarray(req, dtype="float64"); v = np.clip(p_up, 0, 1) * req
    i = np.minimum(np.searchsorted(ladder, v - 1e-9, side="left"), len(ladder) - 1)
    s = np.where(v > ladder[-1] + 1e-9, req, ladder[i]); return np.minimum(s, req) / req

def operating_point(y, req, res_hours, p_up, ladder):
    sf = slice_fraction(p_up, req, ladder)
    return dict(U=float(np.mean(sf < np.asarray(y) - 1e-9)), R_share=float(np.sum(res_hours * (1 - sf)) / np.sum(res_hours)), mean_slice=float(np.mean(sf)))

def frontier(y, req, res, p_base, ladder, margins=None):
    margins = np.linspace(-0.3, 0.7, 51) if margins is None else margins
    pts = [operating_point(y, req, res, np.clip(p_base + m, 0, 1), ladder) for m in margins]
    return pd.DataFrame(pts).assign(margin=margins)

def r_at_u(fr, u_max):
    """Largest reclaimed share among frontier points with U <= u_max (0 if none qualify)."""
    ok = fr[fr["U"] <= u_max]; return float(ok["R_share"].max()) if len(ok) else 0.0

def paired_test(a, b, alternative="greater"):
    """Wilcoxon signed-rank on paired per-fold scores (exact for n <= 25)."""
    a, b = np.asarray(a, float), np.asarray(b, float); d = a - b
    if np.allclose(d, 0): return dict(W=np.nan, p=1.0, n=len(d))
    r = stats.wilcoxon(a, b, alternative=alternative, zero_method="wilcox"); return dict(W=float(r.statistic), p=float(r.pvalue), n=int(len(d)))
print("metrics ready: pinball, conformal_margin, mondrian_margins, upper_bound, slice_fraction, operating_point, frontier, r_at_u, paired_test")

## 6. Design matrix for the linear models (shared by A2 and B1)
Median imputation + a missing-indicator per column that has NaNs (structural missingness carries signal), standardisation, one-hot categoricals with fixed levels.

In [ ]:
class Design:
    def __init__(self, num, cat): self.num, self.cat = list(num), list(cat)
    def fit(self, d):
        X = d[self.num].astype("float64"); self.med = X.median().fillna(0.0)
        self.na_cols = [c for c in self.num if X[c].isna().any()]
        Xi = X.fillna(self.med); self.mu = Xi.mean(); self.sd = Xi.std().replace(0, 1).fillna(1.0)
        self.levels = {c: np.array(list(d[c].cat.categories), dtype=object) for c in self.cat}; return self
    def transform(self, d, dtype="float64"):
        X = d[self.num].astype("float64"); na = X[self.na_cols].isna().astype("float64").values
        Z = ((X.fillna(self.med) - self.mu) / self.sd).clip(-5, 5).values
        oh = [(d[c].astype(str).to_numpy(dtype=object)[:, None] == self.levels[c][None, :]).astype("float64") for c in self.cat]
        return np.hstack([Z, na] + oh).astype(dtype, copy=False)
print("Design matrix helper ready")

## 7. Load D1 (Model A dataset), schema report, fold table

In [ ]:
from IPython.display import display
df = prepare(read_parquet_any(find_path(D1_PATTERN)), "y_mvf", "D1")
schema_report(df)
print("segments in file:", df["segment"].value_counts().to_dict())
df = df[df["segment"].isin(["train", "val", "test"])].reset_index(drop=True)      # plan: warm-up and tail are not part of D1; drop defensively
FEATS = NUM_F + CAT_F                      # transferable features: used for every LOCO model
leakage_gate(FEATS, df)
masks = make_masks(df)
FOLDS = eligible_clusters(df["cluster_id"])
print(f"\n{len(FOLDS)} LOCO folds (clusters with >= {MIN_CLUSTER_PODS:,} D1 pods): {FOLDS}")
print("rows: train %d | val %d (es %d / cal %d) | test %d" % tuple(int(masks[k].sum()) for k in ("train", "val", "val_es", "val_cal", "test")))
FT = fold_table(df, FOLDS); display(FT)
print("cold-start share (no workload history): train %.1f%% | val %.1f%% | test %.1f%%" % tuple(100 * df.loc[masks[k], "cold"].mean() for k in ("train", "val", "test")))

def get_ladder():
    try:
        g = read_parquet_any(find_path(D2_PATTERN), columns=["gpu_request"])["gpu_request"].dropna().astype(float)
        L = np.sort(np.unique(np.round(g, 4))); print(f"deployable slice sizes read from D2: {len(L)} values: {L}")
    except Exception as e:
        L = np.array([.1, .125, .2, .25, .3, .33, .4, .5, .6, .67, .75, .8, 1, 2, 3, 4, 5, 6, 7, 8], float)
        print(f"D2 not readable ({type(e).__name__}); using a GENERIC ladder - set LADDER by hand if you know the real sizes")
    return L
LADDER = get_ladder()
Y = df["y"].values; REQ = df["gpu_request"].values.astype("float64"); RES = df["reserved_gpu_hours"].values.astype("float64")
STRATA = df["stratum"].values
print("target y_mvf: median rec %.3f | genai %.3f" % (df.loc[df.is_rec, "y"].median(), df.loc[~df.is_rec, "y"].median()))

## 8. A1 - the incumbent: per-workload sliding-window percentile (no learning)
For each pod: the p90 of the target over the same workload's earlier pods inside the last *w* days; if the workload has no usable history,
fall back to the p90 of the training data for the same (model_type, gpu_spec). This is the Autopilot-style bar a learned model must clear.

**History audit.** Two rules are computed side by side:
* `completed` (used for A1): only pods whose observation had *finished* before this pod was admitted - information a scheduler could really have had.
* `day` (the plan's literal wording): any pod admitted on an earlier *day*. A pod admitted yesterday has >= 24 h of label, so part of its label lies *after*
  today's pod was admitted, and replicas of one service behave alike. If `day` scores clearly better than `completed`, your `wl_hist_*` columns may leak.

In [ ]:
if FAST: FOLDS = FOLDS[:4]

def window_history(d, windows, q=0.9, rule="completed"):
    n = len(d); out = {w: np.full(n, np.nan) for w in windows}
    wl, y, ta, te, day = d["wl_code"].values, d["y"].values, d["t_adm"].values, d["t_end"].values, d["adm_day_"].values
    idx = np.flatnonzero(wl >= 0); idx = idx[np.argsort(wl[idx], kind="stable")]
    cuts = np.flatnonzero(np.r_[True, wl[idx][1:] != wl[idx][:-1], True])
    for a, b in zip(cuts[:-1], cuts[1:]):
        g = idx[a:b]
        if len(g) < 2: continue
        if rule == "completed":
            o = g[np.argsort(te[g], kind="stable")]; key, ys = te[o], y[o]
            for i in g:
                hi = np.searchsorted(key, ta[i], side="right")
                for w in windows:
                    lo = np.searchsorted(key, ta[i] - 24 * w, side="left")
                    if hi > lo: out[w][i] = np.quantile(ys[lo:hi], q)
        else:
            o = g[np.argsort(day[g], kind="stable")]; key, ys = day[o], y[o]
            for i in g:
                hi = np.searchsorted(key, day[i], side="left")
                for w in windows:
                    lo = np.searchsorted(key, day[i] - w, side="left")
                    if hi > lo: out[w][i] = np.quantile(ys[lo:hi], q)
    return out

t0 = time.time()
for rule, pre in (("completed", "a1_w"), ("day", "a1d_w")):
    H = window_history(df, HIST_WINDOWS, TAU_MAIN, rule)
    for w in HIST_WINDOWS: df[f"{pre}{w}"] = H[w]
print(f"window histories computed in {time.time() - t0:.0f}s")

class A1:
    """Percentile-over-history baseline; window w chosen on validation. The conformal wrapper is applied like for the learned models."""
    name = "A1"
    def __init__(self, w, prefix="a1_w"): self.w, self.prefix = w, prefix
    def fit(self, dtr, *_, tau=TAU_MAIN):
        g = dtr.groupby(["model_type_public", "gpu_spec_public"], observed=True)["y"].quantile(tau)
        self.fb1 = {f"{a}|{b}": v for (a, b), v in g.items()}
        self.fb2 = dtr.groupby("model_type_public", observed=True)["y"].quantile(tau).astype(float).to_dict(); self.fb3 = float(dtr["y"].quantile(tau))
        return self
    def predict(self, d):
        key = d["model_type_public"].astype(str) + "|" + d["gpu_spec_public"].astype(str)
        fb = pd.Series(key.values).map(self.fb1)
        fb = fb.fillna(pd.Series(d["model_type_public"].astype(str).values).map(self.fb2)).fillna(self.fb3).values
        h = d[f"{self.prefix}{self.w}"].values
        return np.where(np.isnan(h), fb, h)

def quick_scores(model, mask):
    d = df[mask]; p = model.predict(d); return pinball(d["y"].values, p, TAU_MAIN), float(np.mean(d["y"].values <= p))

rows = []
for rule, pre in (("completed", "a1_w"), ("day", "a1d_w")):
    for w in HIST_WINDOWS:
        m = A1(w, pre).fit(df[masks["train"]]); pv, cv = quick_scores(m, masks["val"]); pt, ct = quick_scores(m, masks["test"])
        rows.append(dict(rule=rule, window_days=w, val_pinball=pv, val_cov=cv, test_pinball=pt, test_cov=ct,
                         warm_share_val=float(df.loc[masks["val"], f"{pre}{w}"].notna().mean())))
A1_TABLE = pd.DataFrame(rows).round(4); display(A1_TABLE)
BEST_W = int(A1_TABLE[A1_TABLE.rule == "completed"].sort_values("val_pinball").iloc[0].window_days)
print(f"A1 window selected on validation (completed rule): w = {BEST_W} days")
extra = ((df[f"a1d_w{BEST_W}"].notna()) & (df[f"a1_w{BEST_W}"].isna()))[masks["test"]].mean()
gap = A1_TABLE.query("window_days==@BEST_W").set_index("rule")["test_pinball"]
print(f"AUDIT: {extra:.1%} of test pods get history under the 'day' rule but not under 'completed' (only unfinished siblings).")
print(f"       test pinball  completed={gap['completed']:.4f}  day={gap['day']:.4f}  ->  if 'day' is much lower, wl_hist_* built by admission day likely leaks sibling information.")

## 9. A2 - regularised linear quantile regression
Pinball loss at tau, L2 penalty, one-hot categoricals, standardised numerics (median-imputed, with a missing-indicator for every column that has NaNs -
essential here because 60-70% of the history features are structurally missing). Fitted by L-BFGS on a *smoothed* pinball loss
`S(u) = tau*u + a*log(1+exp(-u/a))`, which tends to the exact check loss as a -> 0 and, unlike sklearn's `QuantileRegressor`, scales to ~200k rows.
What it isolates: *is the relationship non-linear at all?*

In [ ]:
from scipy import optimize
from scipy.special import expit

def fit_linear_quantile(X, y, tau, lam, a=0.02, maxiter=300):
    n, p = X.shape
    def fun(th):
        w, b = th[:-1], th[-1]; u = y - X @ w - b; z = -u / a
        loss = np.mean(tau * u + a * np.logaddexp(0.0, z)) + 0.5 * lam * (w @ w)
        s = tau - expit(z)                                   # dS/du
        return loss, np.r_[-(X.T @ s) / n + lam * w, -np.mean(s)]
    th0 = np.zeros(p + 1); th0[-1] = np.quantile(y, tau)
    return optimize.minimize(fun, th0, jac=True, method="L-BFGS-B", options=dict(maxiter=maxiter)).x

class A2:
    name = "A2"
    def __init__(self, lam, tau=TAU_MAIN, num=None, cat=None): self.lam, self.tau, self.num, self.cat = lam, tau, num or NUM_F, CAT_F if cat is None else cat
    def fit(self, dtr, *_):
        self.des = Design(self.num, self.cat).fit(dtr); X = self.des.transform(dtr)
        self.th = fit_linear_quantile(X, dtr["y"].values, self.tau, self.lam); return self
    def predict(self, d):
        X = self.des.transform(d); return np.clip(X @ self.th[:-1] + self.th[-1], 0, 1)

A2_LAMBDAS = (1e-4, 1e-3, 1e-2)                      # equal budget: 3 configurations per family, selected on the held-out validation half

## 10. A3 - gradient-boosted quantile trees (primary model), XGBoost
`reg:quantileerror` at tau = 0.90 with **native categoricals, native NaN handling and monotone constraints** on the history features
(more historical demand can never lower the recommended slice). Early stopping uses the validation half *es*; model selection uses the
disjoint half *cal*; three depth configurations = the same budget as A1 (3 windows) and A2 (3 penalties).

In [ ]:
N_EST, ES_ROUNDS = (150, 20) if FAST else (900, 40)
A3_CFGS = [dict(depth=4, mcw=5), dict(depth=6, mcw=10), dict(depth=8, mcw=20)]

class A3:
    name = "A3"
    def __init__(self, cfg, tau=TAU_MAIN, feats=None, mono=True, lr=0.06):
        self.cfg, self.tau, self.feats, self.mono, self.lr = cfg, tau, list(feats or FEATS), mono, lr
    def fit(self, dtr, des):
        mc = tuple(MONO.get(c, 0) if self.mono else 0 for c in self.feats)
        self.m = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=self.tau, tree_method="hist", enable_categorical=True,
                                  max_cat_to_onehot=1, learning_rate=self.lr, max_depth=self.cfg["depth"], min_child_weight=self.cfg["mcw"],
                                  subsample=0.8, colsample_bytree=0.8, n_estimators=N_EST, early_stopping_rounds=ES_ROUNDS,
                                  monotone_constraints=mc, n_jobs=N_JOBS, random_state=SEED)
        self.m.fit(dtr[self.feats], dtr["y"].values, eval_set=[(des[self.feats], des["y"].values)], verbose=False); return self
    def predict(self, d): return np.clip(self.m.predict(d[self.feats]), 0, 1)

def tune(make, grid, label, train_m, es_m, sel_m):
    best, rows = None, []
    for g in grid:
        t0 = time.time(); m = make(g).fit(df[train_m], df[es_m]); p = m.predict(df[sel_m]); s = pinball(df.loc[sel_m, "y"].values, p, TAU_MAIN)
        rows.append(dict(model=label, config=str(g), sel_pinball=round(s, 5), seconds=round(time.time() - t0, 1)))
        if best is None or s < best[0]: best = (s, g)
    display(pd.DataFrame(rows)); return best[1]

BEST_A2 = tune(lambda lam: A2(lam), A2_LAMBDAS, "A2", masks["train"], masks["val_es"], masks["val_cal"])
BEST_A3 = tune(lambda cfg: A3(cfg), A3_CFGS, "A3", masks["train"], masks["val_es"], masks["val_cal"])
print("selected on the validation half  ->  A1: w =", BEST_W, "| A2: lambda =", BEST_A2, "| A3:", BEST_A3)

## 11. Scoring harness and the temporal (seen-clusters) evaluation
`score_split` calibrates on the calibration half, then reports on the test rows for three variants: **raw** (the tau = 0.9 quantile as is),
**global conformal** (one margin) and **Mondrian conformal** (margin per model_type x cold/warm). `U` is measured after snapping up to the deployable sizes.

In [ ]:
STORE = {}          # (model, fold) -> arrays kept for frontier plots / calibration diagnostics
def score_split(name, fold, model, cal_m, te_m, tau=TAU_MAIN):
    dc, dt = df[cal_m], df[te_m]; yc, yt = dc["y"].values, dt["y"].values
    pc, pt = model.predict(dc), model.predict(dt)
    g, per = mondrian_margins(yc, pc, STRATA[cal_m], ALPHA)
    out = dict(model=name, fold=fold, tau=tau, n_test=int(te_m.sum()), pinball=pinball(yt, pt, tau), cov_raw=float(np.mean(yt <= pt)))
    for tag, up in (("raw", np.clip(pt, 0, 1)), ("glob", upper_bound(pt, g=g)), ("mond", upper_bound(pt, STRATA[te_m], g, per))):
        op = operating_point(yt, REQ[te_m], RES[te_m], up, LADDER)
        out.update({f"U_{tag}": op["U"], f"R_{tag}": op["R_share"], f"cov_{tag}": float(np.mean(yt <= up))})
    out["margin_glob"] = g
    STORE[(name, fold)] = dict(pc=pc, yc=yc, pt=pt, yt=yt, req=REQ[te_m], res=RES[te_m], strata=STRATA[te_m], per=per, g=g)
    return out

def strata_table(name, fold):
    s = STORE[(name, fold)]; up = upper_bound(s["pt"], s["strata"], s["g"], s["per"]); rows = []
    for st in np.unique(s["strata"]):
        m = s["strata"] == st
        if m.sum() >= 30:
            op = operating_point(s["yt"][m], s["req"][m], s["res"][m], up[m], LADDER)
            rows.append(dict(model=name, fold=fold, stratum=st, n=int(m.sum()), coverage=float(np.mean(s["yt"][m] <= up[m])), U=op["U"], R_share=op["R_share"]))
    return pd.DataFrame(rows)

t0 = time.time()
M = dict(A1=A1(BEST_W).fit(df[masks["train"]]), A2=A2(BEST_A2).fit(df[masks["train"]]), A3=A3(BEST_A3).fit(df[masks["train"]], df[masks["val_es"]]))
rows = [score_split(k, "temporal", m, masks["val_cal"], masks["test"]) for k, m in M.items()]
for tau in (0.5, 0.95):                                       # extra base quantiles to trace the risk-savings frontier (plan 6.1)
    mt = A3(BEST_A3, tau=tau).fit(df[masks["train"]], df[masks["val_es"]]); rows.append(score_split(f"A3_tau{tau}", "temporal", mt, masks["val_cal"], masks["test"], tau))
TEMPORAL = pd.DataFrame(rows).set_index("model")
print(f"temporal evaluation ({time.time() - t0:.0f}s): all clusters seen in training, test = days {TEST_DAYS[0]}-{TEST_DAYS[1]}, alpha = {ALPHA}")
display(TEMPORAL[["n_test", "pinball", "cov_raw", "cov_glob", "U_glob", "R_glob", "cov_mond", "U_mond", "R_mond"]].round(4))
display(pd.concat([strata_table(k, "temporal") for k in ("A1", "A2", "A3")]).round(3).set_index(["model", "stratum"]))

## 12. Leave-one-cluster-out: every model on every held-out cluster
Per fold: fit on the *other* clusters' train days, early-stop on their validation half *es*, calibrate conformal margins on their half *cal*,
score the held-out cluster's **test days**. Always read each fold next to its composition (rec share, warm share): error that tracks composition
is not cluster identity (plan 4.3). Expected time on free Colab: roughly 1-3 min per fold; the loop prints its own estimate.

In [ ]:
LOCO_ROWS, t_start = [], time.time()
for k, c in enumerate(FOLDS, 1):
    lm = loco_masks(df, masks, c); t0 = time.time()
    if lm["test"].sum() < 50: print(f"skip {c}: only {lm['test'].sum()} test pods"); continue
    fit = dict(A1=A1(BEST_W).fit(df[lm["train"]]), A2=A2(BEST_A2).fit(df[lm["train"]]), A3=A3(BEST_A3).fit(df[lm["train"]], df[lm["es"]]))
    for name, m in fit.items(): LOCO_ROWS.append(score_split(name, c, m, lm["cal"], lm["test"]))
    el = time.time() - t0; print(f"fold {k}/{len(FOLDS)} held-out {c}: n_test={int(lm['test'].sum())}  {el:.0f}s  (est. remaining {el * (len(FOLDS) - k) / 60:.1f} min)")
LOCO = pd.DataFrame(LOCO_ROWS)
LOCO.to_csv(os.path.join(OUT_DIR, "A_loco_results.csv"), index=False)
comp = FT.loc[[f for f in FOLDS if f in set(LOCO.fold)], ["n_test", "rec_share", "warm_share_test"]]
view = LOCO.pivot(index="fold", columns="model", values=["pinball", "R_mond", "U_mond", "cov_mond"]).round(4)
view.columns = [f"{m}:{metric}" for metric, m in view.columns]                    # flatten (metric, model) -> "A3:pinball"
display(comp.join(view).sort_index(axis=1, key=lambda c: c.str.split(":").str[-1]))
print("mean +- sd over folds (Mondrian conformal, alpha=%.2f):" % ALPHA)
display(LOCO.groupby("model")[["pinball", "cov_raw", "cov_mond", "U_mond", "R_mond"]].agg(["mean", "std"]).round(4))

## 13. GO / NO-GO checkpoint (plan 6.2) and the risk-savings frontier
**Pre-registered rule** (set before looking at results; edit only before running): A3 passes if, over the held-out clusters,
(i) it has lower pinball loss than A1 in at least `WIN_FRAC` of folds, (ii) the one-sided Wilcoxon signed-rank test against A1 has p <= 0.05,
and (iii) at the same conformal level it reclaims more GPU-hours (mean R) than A1 while the mean under-provision rate U stays <= `U_TARGET + 0.02`.
A3 vs A2 is a *secondary* comparison, Holm-corrected together with A2 vs A1, so the "leading pair" is not chosen after the fact.

In [ ]:
WIN_FRAC, U_TARGET = 0.8, 0.05
P = {m: LOCO[LOCO.model == m].set_index("fold") for m in ("A1", "A2", "A3")}
folds_ok = sorted(set(P["A1"].index) & set(P["A3"].index)); n = len(folds_ok)
def pw(a, b, col, alt): return paired_test(P[a].loc[folds_ok, col].values, P[b].loc[folds_ok, col].values, alt)
t_pin = pw("A1", "A3", "pinball", "greater"); t_R = pw("A3", "A1", "R_mond", "greater")      # A3 better than A1 <=> A1 pinball larger / A3 R larger
wins = int((P["A3"].loc[folds_ok, "pinball"].values < P["A1"].loc[folds_ok, "pinball"].values).sum())
r3, r1 = P["A3"].loc[folds_ok, "R_mond"].mean(), P["A1"].loc[folds_ok, "R_mond"].mean(); u3 = P["A3"].loc[folds_ok, "U_mond"].mean()
c_i, c_ii, c_iii = wins >= WIN_FRAC * n, t_pin["p"] <= 0.05, (r3 > r1) and (u3 <= U_TARGET + 0.02)
print(f"A3 vs A1 over {n} held-out clusters")
print(f"  (i)   pinball: A3 better in {wins}/{n} folds (need >= {WIN_FRAC * n:.0f})                         -> {'PASS' if c_i else 'FAIL'}")
print(f"  (ii)  Wilcoxon one-sided p = {t_pin['p']:.4f} (W={t_pin['W']}, n={t_pin['n']}; smallest possible for n={n}: {1 / 2 ** n:.4f})  -> {'PASS' if c_ii else 'FAIL'}")
print(f"  (iii) mean R: A3 {r3:.3f} vs A1 {r1:.3f} (Wilcoxon p = {t_R['p']:.4f}); mean U(A3) = {u3:.3f}          -> {'PASS' if c_iii else 'FAIL'}")
print("\n==> " + ("GO: a learned model clears the incumbent on unseen clusters. Continue to ablations and interpretation." if (c_i and c_ii and c_iii) else
      "NO-GO: no learned model clearly beats the percentile baseline. Per plan 6.2, redirect to the characterisation + pipeline contribution and write the negative result."))
sec = [("A3 vs A2 (pinball)", pw("A2", "A3", "pinball", "greater")["p"]), ("A2 vs A1 (pinball)", pw("A1", "A2", "pinball", "greater")["p"])]
order = np.argsort([p for _, p in sec]); holm = {}; run = 0.0
for rank, i in enumerate(order):
    run = max(run, min(1.0, sec[i][1] * (len(sec) - rank))); holm[sec[i][0]] = run
print("secondary comparisons (one-sided Wilcoxon, Holm-adjusted p):", {k: round(v, 4) for k, v in holm.items()})

# --- figures: per-fold pinball and the frontier (oracle margin sweep on pooled held-out predictions) ---
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
x = np.arange(n); wd = 0.27
for j, m in enumerate(("A1", "A2", "A3")): ax[0].bar(x + (j - 1) * wd, P[m].loc[folds_ok, "pinball"], wd, label=m)
ax[0].set_xticks(x); ax[0].set_xticklabels(folds_ok, rotation=45); ax[0].set_ylabel("pinball loss, tau=0.9 (lower is better)"); ax[0].set_title("Held-out clusters"); ax[0].legend()
for m in ("A1", "A2", "A3"):
    cat = lambda key: np.concatenate([STORE[(m, f)][key] for f in folds_ok])
    fr = frontier(cat("yt"), cat("req"), cat("res"), cat("pt"), LADDER); ax[1].plot(fr["U"], fr["R_share"], label=m)
for tau in (0.5, 0.95):
    s = STORE[(f"A3_tau{tau}", "temporal")]; fr = frontier(s["yt"], s["req"], s["res"], s["pt"], LADDER); ax[1].plot(fr["U"], fr["R_share"], "--", alpha=.6, label=f"A3 tau={tau} (temporal)")
ax[1].axvline(U_TARGET, color="k", ls=":"); ax[1].set_xlim(0, 0.3); ax[1].set_xlabel("U: under-provision rate"); ax[1].set_ylabel("R: reclaimed share of reserved GPU-hours")
ax[1].set_title("Risk-savings frontier (margin swept; dotted = U target)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "A_gonogo_frontier.png"), dpi=FIG_DPI); plt.show()

## 14. Extra analysis: A1 vs A3 at the SAME risk level (exploratory)
The go / no-go above compares models at their own calibrated margins, so a model that takes more risk frees more GPU.
Here every model gets the margin that puts it at exactly the same under-provision rate (`U_MATCH`, default 5%) *in each held-out cluster*,
and we compare how much each frees. The margin is chosen on the test pods themselves (an oracle sweep), so treat this as an
**exploratory, apples-to-apples comparison**, not as a deployable result. It was added after the pre-registered gate and does not change it.

In [ ]:
U_MATCH = 0.05
rows = []
for f in folds_ok:
    r = {"fold": f, "n_test": int(len(STORE[("A3", f)]["yt"]))}
    for m in ("A1", "A2", "A3"):
        s = STORE[(m, f)]; fr = frontier(s["yt"], s["req"], s["res"], s["pt"], LADDER, margins=np.linspace(-0.3, 0.7, 101))
        r[f"{m}_R_at_U{int(U_MATCH * 100)}"] = r_at_u(fr, U_MATCH)
    rows.append(r)
MATCH = pd.DataFrame(rows).set_index("fold"); col = lambda m: f"{m}_R_at_U{int(U_MATCH * 100)}"
MATCH["A3_minus_A1"] = MATCH[col("A3")] - MATCH[col("A1")]
display(MATCH.round(4))
t = paired_test(MATCH[col("A3")].values, MATCH[col("A1")].values, "greater")
print(f"At U = {U_MATCH:.0%} in every held-out cluster: mean reclaimed share  A1 {MATCH[col('A1')].mean():.3f}  |  A2 {MATCH[col('A2')].mean():.3f}  |  A3 {MATCH[col('A3')].mean():.3f}")
print(f"A3 - A1 = {MATCH['A3_minus_A1'].mean():+.3f} (median {MATCH['A3_minus_A1'].median():+.3f}); A3 higher in {(MATCH['A3_minus_A1'] > 0).sum()}/{len(MATCH)} clusters; one-sided Wilcoxon p = {t['p']:.4f}")
MATCH.to_csv(os.path.join(OUT_DIR, "A_matched_risk.csv"))

## 15. Ablations (plan 6.4)
Each variant is A3 refit with one thing changed, scored on the temporal split **and** on the first `ABL_FOLDS` held-out clusters (default: the 3 largest;
widen it if you have time). Reported as the change versus the full model, so a *negative* delta in pinball is an improvement.
Not runnable from D1 alone: the observation-threshold ablation (>= 24 h vs >= 6 h vs unrestricted) needs D1 rebuilt in SQL with the other thresholds.
`cluster_id` / `asw_id` variants are temporal-split only, because under LOCO those ids are unseen by construction.

In [ ]:
ABL_FOLDS = FOLDS[:3]

class SplitBySegment:
    """Pooled-vs-separate ablation: one A3 for rec pods, one for genai pods."""
    name = "A3_sep"
    def fit(self, dtr, des):
        self.m = {s: A3(BEST_A3).fit(dtr[dtr["is_rec"] == s], des[des["is_rec"] == s]) for s in (True, False)}; return self
    def predict(self, d):
        p = np.zeros(len(d)); r = d["is_rec"].values
        for s, m in self.m.items():
            if (r == s).any(): p[r == s] = m.predict(d[r == s])
        return p

VARIANTS = {
    "full":              (lambda: A3(BEST_A3), True),
    "no_workload_history": (lambda: A3(BEST_A3, feats=[f for f in FEATS if f not in HIST_COLS]), True),
    "no_cluster_load":   (lambda: A3(BEST_A3, feats=[f for f in FEATS if f not in CLUSTER_LOAD_COLS]), True),
    "no_monotone":       (lambda: A3(BEST_A3, mono=False), True),
    "separate_rec_genai": (lambda: SplitBySegment(), True),
    "with_cluster_id":   (lambda: A3(BEST_A3, feats=FEATS + [c for c in CLS_F if c == "cluster_id"], mono=True), False),
    "with_asw_id":       (lambda: A3(BEST_A3, feats=FEATS + [c for c in CLS_F if c == "asw_id"], mono=True), False),
}
ABL_ROWS, t0 = [], time.time()
for label, (make, loco_ok) in VARIANTS.items():
    if label.startswith("with_") and not any(c in CLS_F for c in ("cluster_id", "asw_id")): continue
    splits = [("temporal", dict(train=masks["train"], es=masks["val_es"], cal=masks["val_cal"], test=masks["test"]))]
    if loco_ok: splits += [(c, loco_masks(df, masks, c)) for c in ABL_FOLDS]
    for fold, lm in splits:
        m = make().fit(df[lm["train"]], df[lm["es"]]); r = score_split(f"abl:{label}", fold, m, lm["cal"], lm["test"]); r["variant"] = label; ABL_ROWS.append(r)
    print(f"  {label:22s} done ({time.time() - t0:.0f}s elapsed)")
ABL = pd.DataFrame(ABL_ROWS)
ref = ABL[ABL.variant == "full"].set_index("fold")
out = []
for v, g in ABL.groupby("variant", sort=False):
    g = g.set_index("fold"); common = g.index.intersection(ref.index)
    out.append(dict(variant=v, n_splits=len(common), d_pinball=(g.loc[common, "pinball"] - ref.loc[common, "pinball"]).mean(),
                    d_R_mond=(g.loc[common, "R_mond"] - ref.loc[common, "R_mond"]).mean(), U_mond=g.loc[common, "U_mond"].mean(), cov_mond=g.loc[common, "cov_mond"].mean()))
ABL_TABLE = pd.DataFrame(out).set_index("variant").round(4); ABL_TABLE.to_csv(os.path.join(OUT_DIR, "A_ablations.csv")); display(ABL_TABLE)

## 16. Interpretation (plan section 8): by construction, then global
* **Monotone audit**: proves the constraint holds on real rows (violation rate must print 0.0), so "more history never lowers the slice" is a fact, not a claim.
* **Sign check**: within-segment Spearman of candidate features with the target, to decide with evidence whether any other feature deserves a constraint.
* **Global attribution**: split-gain importance, **permutation importance on a held-out cluster** (generalising signal, not training fit), and SHAP.

In [ ]:
A3T = M["A3"]; BOOST = A3T.m.get_booster(); IT = (0, A3T.m.best_iteration + 1)
dt = df[masks["test"]]

def mono_violation_rate(model, d, feat, n=1500):
    base = d.sample(min(n, len(d)), random_state=SEED).copy(); vals = d[feat].dropna()
    if vals.nunique() < 2: return np.nan
    P = []
    for v in np.unique(np.quantile(vals, np.linspace(0.01, 0.99, 25))):
        b = base.copy(); b[feat] = np.float32(v); P.append(model.predict(b))
    return float((np.diff(np.vstack(P), axis=0) < -1e-6).any(axis=0).mean())
print("monotone audit - share of rows whose prediction ever DEcreases as the feature rises (must be 0):")
print({f: mono_violation_rate(A3T, dt, f) for f in MONO if f in FEATS})

tr = df[masks["train"]]; sign_rows = []
for f in [c for c in ["cpu_per_gpu", "gpu_mem_per_gpu", "vram_entitled_gib", "gpu_request", "server_gpu_occupancy_at_submit", "cluster_sm_mean_prev_hr"] if c in FEATS]:
    row = {"feature": f}
    for seg, sm in (("rec", tr["is_rec"]), ("genai", ~tr["is_rec"])):
        v = tr.loc[sm, [f, "y"]].dropna(); row[f"spearman_{seg}"] = round(float(stats.spearmanr(v[f], v["y"])[0]), 3) if len(v) > 100 and v[f].nunique() > 1 else np.nan
    v = tr[[f, "y"]].dropna(); row["spearman_pooled"] = round(float(stats.spearmanr(v[f], v["y"])[0]), 3); sign_rows.append(row)
print("\nsign check (train rows). Constrain a feature only if the sign is the same, and not tiny, in BOTH segments:"); display(pd.DataFrame(sign_rows).set_index("feature"))

gain = pd.Series(BOOST.get_score(importance_type="total_gain")).sort_values(ascending=False)
c0 = FOLDS[0]; lm0 = loco_masks(df, masks, c0); M_LOCO0 = A3(BEST_A3).fit(df[lm0["train"]], df[lm0["es"]])
dho = df[lm0["test"]]; dho = dho.sample(min(20000, len(dho)), random_state=SEED); base = pinball(dho["y"].values, M_LOCO0.predict(dho), TAU_MAIN)
perm = {}
for f in FEATS:
    inc = []
    for r in range(3):
        d2 = dho.copy(); d2[f] = d2[f].sample(frac=1, random_state=SEED + r).values; inc.append(pinball(d2["y"].values, M_LOCO0.predict(d2), TAU_MAIN) - base)
    perm[f] = (np.mean(inc), np.std(inc))
PERM = pd.DataFrame(perm, index=["increase_in_pinball", "sd"]).T.sort_values("increase_in_pinball", ascending=False)

Xs = dt[FEATS].sample(min(5000, len(dt)), random_state=SEED)
PHI = BOOST.predict(xgb.DMatrix(Xs, enable_categorical=True), pred_contribs=True, iteration_range=IT)
print(f"SHAP additivity check (max |sum(phi) - prediction|): {np.abs(PHI.sum(1) - BOOST.predict(xgb.DMatrix(Xs, enable_categorical=True), iteration_range=IT)).max():.2e}")
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
gain.head(15)[::-1].plot.barh(ax=ax[0], title="split gain (temporal model)")
PERM["increase_in_pinball"].head(15)[::-1].plot.barh(ax=ax[1], xerr=PERM["sd"].head(15)[::-1], title=f"permutation importance, held-out cluster {c0}")
pd.Series(np.abs(PHI[:, :-1]).mean(0), index=FEATS).sort_values().tail(15).plot.barh(ax=ax[2], title="mean |SHAP| (test sample)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "A_importance.png"), dpi=FIG_DPI); plt.show()
try:
    import shap
    Xn = Xs.copy()
    for c in CAT_F: Xn[c] = Xn[c].cat.codes
    shap.summary_plot(PHI[:, :-1], Xn, max_display=15, show=False); plt.savefig(os.path.join(OUT_DIR, "A_shap_summary.png"), dpi=FIG_DPI, bbox_inches="tight"); plt.show()
except Exception as e:
    print("shap beeswarm skipped:", type(e).__name__, e)

## 17. Per-decision explanations, a distilled rule set, and effect shapes
* Each recommendation becomes one audit-log sentence: `f(x) = phi_0 + sum_j phi_j(x)` (SHAP, in target units), plus the conformal margin and the snap-up.
* A **depth-3 tree fitted to A3's predictions** gives rules a reviewer can read without running code; its **fidelity** (R^2 to A3, on rows it did not see) says how far to trust them.
  Missing history is encoded as -1, so a rule like `wl_hist_mvf_p90 <= -0.5` means "no workload history".
* **PDP** (with ICE lines) and **ALE** for the three numeric features that matter most on the held-out cluster. ALE is the safer read when features are correlated.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import r2_score
S_T = STORE[("A3", "temporal")]
def _fmt(v):
    try: return "none" if pd.isna(v) else (f"{float(v):.3g}" if not isinstance(v, str) else v)
    except (TypeError, ValueError): return str(v)

def explain_row(i):
    x, phi, bias = Xs.iloc[i], PHI[i, :-1], PHI[i, -1]; row = dt.loc[Xs.index[i]]
    yhat = float(np.clip(PHI[i].sum(), 0, 1)); margin = S_T["per"].get(row["stratum"], S_T["g"]); req = float(row["gpu_request"])
    sf = slice_fraction(np.array([np.clip(yhat + margin, 0, 1)]), np.array([req]), LADDER)[0]
    parts = [f"{FEATS[j]}={_fmt(x.iloc[j])} ({phi[j]:+.2f})" for j in np.argsort(-np.abs(phi))[:4]]
    return (f"Recommend {sf * req:g} of {req:g} GPU (fraction {sf:.2f}). Predicted need {yhat:.2f} = base {bias:.2f}; " + ", ".join(parts)
            + f"; {margin:+.2f} conformal margin [{row['stratum']}]; snapped up to a deployable size. (observed label {row['y']:.2f})")
for i in range(min(5, len(Xs))): print("-", explain_row(i))

Xd = pd.get_dummies(dt[FEATS], columns=CAT_F, dtype=float).fillna(-1.0)
yd = A3T.predict(dt); half = np.random.default_rng(SEED).random(len(Xd)) < 0.5
SURR = DecisionTreeRegressor(max_depth=3, min_samples_leaf=max(50, len(Xd) // 200), random_state=SEED).fit(Xd[half], yd[half])
print(f"\ndistilled depth-3 rules - fidelity to A3 on unseen rows: R^2 = {r2_score(yd[~half], SURR.predict(Xd[~half])):.3f}")
print(export_text(SURR, feature_names=list(Xd.columns), decimals=3))

def pdp_ice(model, d, f, n=1500, n_ice=30):
    base = d.sample(min(n, len(d)), random_state=SEED); grid = np.unique(np.quantile(d[f].dropna(), np.linspace(0.02, 0.98, 20)))
    P = []
    for v in grid: b = base.copy(); b[f] = np.float32(v); P.append(model.predict(b))
    P = np.vstack(P); return grid, P.mean(1), P[:, :n_ice]

def ale_1d(model, d, f, K=20):
    d = d[d[f].notna()]; x = d[f].values.astype(float); edges = np.unique(np.quantile(x, np.linspace(0, 1, K + 1)))
    if len(edges) < 3: return None, None
    idx = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, len(edges) - 2)
    lo, hi = d.copy(), d.copy(); lo[f] = edges[idx].astype("float32"); hi[f] = edges[idx + 1].astype("float32")
    diff = model.predict(hi) - model.predict(lo); cnt = np.bincount(idx, minlength=len(edges) - 1)
    eff = np.array([diff[idx == b].mean() if cnt[b] else 0.0 for b in range(len(edges) - 1)])
    ale = np.r_[0, np.cumsum(eff)]; return edges, ale - np.average((ale[:-1] + ale[1:]) / 2, weights=np.maximum(cnt, 1))

TOP_NUM = [f for f in PERM.index if f in NUM_F and df[f].nunique() > 2][:3]
fig, ax = plt.subplots(2, max(1, len(TOP_NUM)), figsize=(5 * max(1, len(TOP_NUM)), 7), squeeze=False)
for j, f in enumerate(TOP_NUM):
    g, mean, ice = pdp_ice(A3T, dt.sample(min(20000, len(dt)), random_state=SEED), f)
    ax[0, j].plot(g, ice, color="grey", alpha=.25, lw=.7); ax[0, j].plot(g, mean, lw=2.5); ax[0, j].set_title(f"PDP + ICE: {f}"); ax[0, j].set_ylabel("predicted slice fraction")
    e, a = ale_1d(A3T, dt.sample(min(20000, len(dt)), random_state=SEED), f)
    if e is not None: ax[1, j].plot(e, a, marker="o", ms=3); ax[1, j].axhline(0, color="k", lw=.5)
    ax[1, j].set_title(f"ALE: {f}"); ax[1, j].set_xlabel(f)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "A_effects.png"), dpi=FIG_DPI); plt.show()

## 18. Inference latency and saved artefacts
A scheduler plugin scores one pod at a time, so the number that matters is **single-call latency** (plan: single-digit milliseconds).
Colab CPUs are slower than a production node, so treat these as upper bounds. The saved JSON files are what notebook B loads to compose A with B.

In [ ]:
one = dt[FEATS]; lat = []
for i in range(300):
    r = one.iloc[[i % len(one)]]; t0 = time.perf_counter(); A3T.m.predict(r); lat.append((time.perf_counter() - t0) * 1e3)
t0 = time.perf_counter(); A3T.m.predict(one.iloc[:10000]); batch_us = (time.perf_counter() - t0) / min(10000, len(one)) * 1e6
print(f"A3 single-call latency: p50 {np.percentile(lat, 50):.2f} ms | p95 {np.percentile(lat, 95):.2f} ms | p99 {np.percentile(lat, 99):.2f} ms; batched: {batch_us:.1f} us/pod")
print(f"trees used: {A3T.m.best_iteration + 1} | depth {BEST_A3['depth']}")

A3T.m.save_model(os.path.join(OUT_DIR, "modelA_A3_tau0.90.json"))
meta = dict(features=FEATS, categorical=CAT_F, categories={c: [str(v) for v in df[c].cat.categories] for c in CAT_F}, monotone=MONO,
            config=BEST_A3, tau=TAU_MAIN, alpha=ALPHA, ladder=[float(v) for v in LADDER], margin_global=float(S_T["g"]),
            margin_by_stratum={k: float(v) for k, v in S_T["per"].items()}, a1_window_days=BEST_W, a2_lambda=BEST_A2,
            trained_on=f"days {TRAIN_DAYS[0]}-{TRAIN_DAYS[1]}, all clusters; conformal calibrated on the validation half 'cal'")
json.dump(meta, open(os.path.join(OUT_DIR, "modelA_meta.json"), "w"), indent=1)
TEMPORAL.to_csv(os.path.join(OUT_DIR, "A_temporal_results.csv")); PERM.to_csv(os.path.join(OUT_DIR, "A_permutation_importance.csv"))
print("saved to", OUT_DIR, ":", sorted(f for f in os.listdir(OUT_DIR) if f.startswith(("A_", "modelA"))))